# Example 10.4.3 / 1: Exact PWL Terms versus Approximated Functions

Companion notebook for **10.4.3 Automatic Piecewise-Linear Approximation** in the AMPL lab Documentation, section "Exact PWL Terms versus Approximated Functions". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exact PWL Terms versus Approximated Functions

Two different things can reach a linear solver as piecewise-linear structure:

| What the model contains | What the solver receives | Exact? |
|---|---|---|
| A PWL term you wrote, `<<breakpoints; slopes>> Hours` | That function, linearized by AMPL or by the MP driver, or passed natively | Yes: the PWL function *is* the model |
| A nonlinear function of a variable, `log(1 + Make)`, with a linear-quadratic MP solver | A PWL function built by the MP driver from sample points | No: an approximation with a controlled error |
| A nonlinear function of a variable, with a solver that accepts it natively (Gurobi's `exp`, `log`, `sin`...) | The function itself, as an expression tree | Yes, unless you ask the solver to approximate it |
| A nonlinear function of data only, `log(budget)` | A number, computed by AMPL before the solve | Yes |

In the first row the breakpoints are your modeling decision and the PWL graph is the intended
function, so the reformulation guarantees of
Convexity and Piecewise-Linear Reformulations (AMPL lab documentation, section 10.4.1) apply. In the second row the
breakpoints are chosen by the driver, the PWL graph differs from the true curve between them, and
a "solved" status certifies only the approximate model.

For an exact PWL term, MP's documentation recommends AMPL option `pl_linearize` set to 0 for MP
solvers (its default is 1). AMPL then sends the term as an expression, and the MP driver uses the
solver's native PWL support (Gurobi, COPT) or linearizes it itself (HiGHS). Here a volume discount
makes the cost concave, which is nonconvex to minimize, so the MP driver chooses an exact
formulation with discrete structure where needed:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param required_hours >= 0;  # hours of work needed
var Hours >= required_hours, <= 100;  # hours worked; finite bound

# 35 per hour for the first 40 hours, 20 per hour beyond
minimize LaborCost: <<40; 35, 20>> Hours;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["required_hours"] = 60

In [5]:
# run.py
ampl.option["pl_linearize"] = 0          # pass the PWL term to the MP driver
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
print(ampl.var["Hours"].value(), ampl.obj["LaborCost"].value())

solved
60.0 1800.0


### Pure AMPL: data.dat and model.run

Pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param required_hours := 60;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option pl_linearize 0;
option solver highs;
option solver_msg 0;
solve;
display Hours, LaborCost;

Hours = 60
LaborCost = 1800

